# DENTEX Dataset Preparation

This notebook constructs the canonical dataset used by the Dental X-ray Pathology Detection project. It begins with the original DENTEX Challenge 2023 files, identifies the fully annotated diagnosis subset, validates the selected images and annotations, creates a reproducible train/validation/test split, and writes a simplified COCO-compatible representation for the four-class object-detection task.

The procedure deliberately separates source data from derived data. The contents of `raw/` are treated as immutable; all generated images, annotations, and split metadata are stored under `data/processed/dentex_diagnosis/`.


## 1. Dataset Source and Scope

The project uses [DENTEX Challenge 2023](https://www.kaggle.com/datasets/truthisneverlinear/dentex-challenge-2023), which contains panoramic dental radiographs and annotations for several related computer-vision tasks. DENTEX has a hierarchical structure and was not designed solely as a diagnosis-category dataset. Its annotation levels correspond to the following subtasks:

- **quadrant detection:** identifying the dental quadrant in which a tooth is located;
- **tooth enumeration:** identifying the tooth number;
- **diagnosis detection:** localising a tooth or relevant region and assigning a diagnostic category;
- **unlabeled data:** images without annotations, which cannot be used directly for supervised detector training.

This hierarchy is reflected in the original filesystem. The training data is divided into subsets according to the annotation level available: quadrant-only data, quadrant and tooth-enumeration data, fully annotated diagnosis data, and unlabeled images. The dataset also includes a separate official validation subset.

The annotations use a COCO-like JSON organisation. Each file contains image records, object annotations, and category definitions. An image record provides an identifier, filename, width, and height. Each object annotation is linked to its image through `image_id` and contains a bounding box together with category labels.

In the original DENTEX annotation, a bounding box is represented as

$$
\mathbf{b} = (x, y, w, h),
$$

where $x$ and $y$ are the coordinates of the upper-left corner, while $w$ and $h$ are the box width and height. Thus, unlike the corner-coordinate representation $(x_1, y_1, x_2, y_2)$ often used when formulating an object-detection task, DENTEX stores boxes in COCO form as $[x, y, w, h]$.

Because DENTEX is hierarchical, a fully annotated object may contain several category levels. In particular, the source JSON uses `category_id_1`, `category_id_2`, and `category_id_3` for different levels of the original task. This project requires the third level, `category_id_3`, because it represents the diagnosis category.

| Class ID | Diagnosis category |
|---:|---|
| 0 | Impacted |
| 1 | Caries |
| 2 | Periapical Lesion |
| 3 | Deep Caries |

The project is specifically concerned with detecting these four categories. Quadrant-only and tooth-enumeration subsets do not contain the required target variable. Unlabeled images likewise cannot be used directly for supervised training or quantitative model evaluation. The selected source is therefore the fully annotated diagnosis subset, which provides images, bounding boxes, and diagnosis categories for 705 panoramic radiographs. These 705 images form the source population from which the project-specific train, validation, and test subsets are created.

The official DENTEX validation subset contains 50 images, but the available directory structure does not provide a JSON file with ground-truth annotations for them. Without reference annotations, detection metrics cannot be computed, so these images cannot serve as a complete quantitative validation or test subset in this project.


## 2. Configuration and Dataset Discovery

The preparation workflow uses project-relative paths so that it can run from the repository root or from `notebooks/` without embedding machine-specific Windows paths. A fixed random seed is declared centrally for the deterministic split procedure.

Two data locations have different responsibilities:

- `raw/` is the immutable source dataset. The notebook reads from it but never edits, renames, moves, or deletes its contents.
- `data/processed/dentex_diagnosis/` is the derived canonical dataset used by later notebooks and experiments.

The required DENTEX subset is not selected by assuming a particular folder name. Instead, the notebook searches the available annotation JSON files and identifies the diagnosis subset from the categories present in their contents. This is more robust to changes in directory spelling or nesting and ensures that selection is based on annotation semantics.


In [1]:
from collections import Counter, defaultdict
import json
from pathlib import Path
import random
import shutil
import warnings

import pandas as pd

RANDOM_SEED = 42
EXPECTED_CATEGORIES = {
    0: "Impacted",
    1: "Caries",
    2: "Periapical Lesion",
    3: "Deep Caries",
}
SPLIT_SIZES = {"train": 493, "val": 106, "test": 106}
LABEL_COLUMNS = [
    "has_impacted",
    "has_caries",
    "has_periapical_lesion",
    "has_deep_caries",
]

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "raw").is_dir() else cwd.parent
RAW_ROOT = PROJECT_ROOT / "raw"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "dentex_diagnosis"

if not RAW_ROOT.is_dir():
    raise FileNotFoundError(f"Raw dataset directory not found: {RAW_ROOT}")

configuration = pd.DataFrame(
    {
        "Value": [PROJECT_ROOT, RAW_ROOT, PROCESSED_ROOT, RANDOM_SEED],
    },
    index=["Project root", "Immutable source", "Derived output", "Random seed"],
)
display(configuration)

,Value
Project root,C:\Users\Eclipse\PycharmProjects\Dental-Xray-P...
Immutable source,C:\Users\Eclipse\PycharmProjects\Dental-Xray-P...
Derived output,C:\Users\Eclipse\PycharmProjects\Dental-Xray-P...
Random seed,42


## 3. Original Dataset Inventory

Before selecting data, the notebook inventories the relevant source directories directly from the filesystem. The summary reports where annotation JSON files occur and how many PNG images are available under each corresponding location. Counts are computed from the files currently present rather than copied from documentation or hard-coded.

This inventory makes the original hierarchy visible without printing thousands of individual filenames. It also distinguishes subsets that provide annotations from image-only locations and establishes why only one part of the source dataset is appropriate for the diagnosis-level detector. The “Used by this notebook” column indicates that the final decision is made by inspecting annotation content in the next section, rather than by relying on the directory label alone.


In [2]:
def discover_dataset_inventory(raw_root: Path) -> pd.DataFrame:
    """Summarise annotated DENTEX directories without listing individual images."""
    rows = []
    annotation_directories = sorted(
        {path.parent for path in raw_root.rglob("*.json")},
        key=lambda path: path.as_posix(),
    )
    for directory in annotation_directories:
        rows.append(
            {
                "Directory": str(directory.relative_to(raw_root)),
                "Annotation JSON files": len(list(directory.glob("*.json"))),
                "PNG images": sum(1 for _ in directory.rglob("*.png")),
                "Selection method": "Annotation content",
            }
        )
    return pd.DataFrame(rows)

In [3]:
dataset_inventory = discover_dataset_inventory(RAW_ROOT)
display(dataset_inventory)

,Directory,Annotation JSON files,PNG images,Selection method
0,training_data\training_data\quadrant,1,693,Annotation content
1,training_data\training_data\quadrant-enumerati...,1,705,Annotation content
2,training_data\training_data\quadrant_enumeration,1,634,Annotation content


## 4. Diagnosis Subset Selection

The next step programmatically separates the required diagnosis data from the other DENTEX tasks. The notebook examines each available annotation JSON and extracts its category mapping. A file is considered a diagnosis candidate only when its mapping contains all four expected target classes with their canonical IDs:

$$
\mathcal{C} = \{\text{Impacted},\ \text{Caries},\ \text{Periapical Lesion},\ \text{Deep Caries}\}.
$$

The selection must be unambiguous. Exactly one JSON file is expected to match the diagnosis-category mapping. If no file matches, the required source data is unavailable or structurally different from what the project expects. If several files match, the notebook cannot safely infer which one is canonical. In either situation, preparation stops instead of risking the use of annotations from the wrong DENTEX subtask.

After the unique JSON is identified, its sibling `xrays/` directory is selected as the physical image source. The resulting input consists of two linked components: a JSON file containing image and object records, and a directory containing the corresponding PNG radiographs.

Conceptually, the selected dataset is

$$
D = \{(I_i, A_i)\}_{i=1}^{705},
$$

where $I_i$ is the $i$-th panoramic radiograph and $A_i$ is the set of objects annotated on it:

$$
A_i = \{(\mathbf{b}_{ij}, c_{ij})\}_{j=1}^{N_i},
\qquad c_{ij} \in \mathcal{C}.
$$

Here, $N_i$ is the number of annotated objects in image $I_i$, $\mathbf{b}_{ij}$ is the bounding box of object $j$, and $c_{ij}$ is its diagnosis class. The number of objects differs across images, and a single panoramic radiograph may contain objects from several target classes. Both properties matter when the data is later divided into train, validation, and test subsets.

At this stage, the 705 images are not yet split. Image–annotation links, identifiers, class labels, and bounding boxes must first be validated, and the hierarchical annotation records must be normalised to a single project-specific COCO-compatible schema.


In [4]:
def category_mapping(payload: dict) -> dict[int, str] | None:
    """Return the canonical diagnosis mapping when it is present in a payload."""
    for key, value in payload.items():
        if key.startswith("categories") and isinstance(value, list):
            mapping = {int(item["id"]): str(item["name"]) for item in value}
            if mapping == EXPECTED_CATEGORIES:
                return mapping
    return None


def find_diagnosis_subset(
    raw_root: Path,
    expected_categories: dict[int, str],
) -> tuple[Path, Path, dict]:
    """Find the unique JSON whose diagnosis categories match the project task."""
    candidates = []
    for annotation_path in sorted(raw_root.rglob("*.json")):
        with annotation_path.open("r", encoding="utf-8") as handle:
            payload = json.load(handle)
        if category_mapping(payload) == expected_categories:
            candidates.append((annotation_path, payload))

    if len(candidates) != 1:
        paths = [str(path) for path, _ in candidates]
        raise RuntimeError(
            f"Expected exactly one diagnosis annotation file; found {len(candidates)}: {paths}"
        )

    source_json, source_payload = candidates[0]
    source_image_dir = source_json.parent / "xrays"
    return source_json, source_image_dir, source_payload

In [5]:
SOURCE_JSON, SOURCE_IMAGE_DIR, source_payload = find_diagnosis_subset(
    RAW_ROOT,
    EXPECTED_CATEGORIES,
)

images = source_payload.get("images", [])
annotations = source_payload.get("annotations", [])
categories = category_mapping(source_payload)

selected_subset = pd.DataFrame(
    {
        "Value": [
            SOURCE_JSON.relative_to(PROJECT_ROOT),
            SOURCE_IMAGE_DIR.relative_to(PROJECT_ROOT),
            len(images),
            len(annotations),
        ]
    },
    index=["Selected annotation JSON", "Selected image directory", "Images", "Annotations"],
)
display(selected_subset)

,Value
Selected annotation JSON,raw\training_data\training_data\quadrant-enume...
Selected image directory,raw\training_data\training_data\quadrant-enume...
Images,705
Annotations,3529


## 5. Integrity Validation and Annotation Semantics

Validation is performed before any split is constructed. The purpose is to ensure that the JSON structure agrees with the physical image files and that every object has a valid class and bounding box before source errors can propagate into processed data.

Critical inconsistencies are handled with fail-fast checks. The notebook does not silently correct, discard, or reinterpret invalid annotations. Automatic repair could conceal a source-data problem and make the resulting dataset difficult to audit. Instead, preparation stops with an explicit error so that the inconsistency can be investigated.

First, every `file_name` in the JSON `images` section must correspond to an existing PNG in the selected diagnosis directory. This prevents annotations from being retained for images that are physically absent. Image-record IDs must also be unique, as must annotation IDs; duplicate identifiers could make image–annotation linkage ambiguous during later loading and evaluation.

Each annotation's `image_id` must reference a real record in the `images` section. Referential integrity is expressed as

$$
\texttt{annotation.image\_id}
\in
\{\texttt{image.id} \mid \texttt{image} \in \mathrm{Images}\}.
$$

The diagnosis label is read from `category_id_3`. Only the four project class IDs are permitted:

$$
\texttt{category\_id\_3} \in \{0,1,2,3\}.
$$

Validation checks both that no unknown class ID occurs and that every expected class has at least one annotation.

Bounding boxes receive a separate geometric check. DENTEX stores each source box in COCO form,

$$
\mathbf{b} = (x, y, w, h),
$$

where $(x,y)$ is the upper-left corner and $w$ and $h$ are width and height. Every `bbox` must therefore contain exactly four numeric values and satisfy

$$
x \geq 0, \qquad y \geq 0, \qquad w > 0, \qquad h > 0.
$$

For an image of width $W$ and height $H$, the box must also remain within image boundaries:

$$
x+w \leq W, \qquad y+h \leq H.
$$

The implementation allows only a tolerance of $10^{-6}$ in boundary comparisons to accommodate floating-point representation. The compact validation report retains visible evidence for file counts, identifiers, references, class IDs, and bounding-box checks; any non-zero critical-error count prevents the workflow from continuing.


In [6]:
def _bbox_is_valid(bbox, image: dict, tolerance: float = 1e-6) -> bool:
    if not isinstance(bbox, list) or len(bbox) != 4:
        return False
    try:
        x, y, width, height = map(float, bbox)
    except (TypeError, ValueError):
        return False
    return (
        x >= -tolerance
        and y >= -tolerance
        and width > 0
        and height > 0
        and x + width <= float(image["width"]) + tolerance
        and y + height <= float(image["height"]) + tolerance
    )


def validate_source_dataset(
    payload: dict,
    source_image_dir: Path,
    expected_categories: dict[int, str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fail on structural errors and return compact source-validation tables."""
    images = payload.get("images", [])
    annotations = payload.get("annotations", [])
    categories = category_mapping(payload)
    image_ids = [int(image["id"]) for image in images]
    annotation_ids = [int(annotation["id"]) for annotation in annotations]
    image_by_id = {int(image["id"]): image for image in images}

    missing_files = [
        image["file_name"]
        for image in images
        if not (source_image_dir / image["file_name"]).is_file()
    ]
    unknown_image_refs = []
    invalid_boxes = []
    unexpected_class_ids = []

    for annotation in annotations:
        image_id = int(annotation["image_id"])
        class_id = int(annotation["category_id_3"])
        if image_id not in image_by_id:
            unknown_image_refs.append(annotation["id"])
            continue
        if class_id not in expected_categories:
            unexpected_class_ids.append((annotation["id"], class_id))
        if not _bbox_is_valid(annotation.get("bbox"), image_by_id[image_id]):
            invalid_boxes.append((annotation["id"], annotation.get("bbox")))

    checks = {
        "PNG files": len(list(source_image_dir.glob("*.png"))),
        "JSON image records": len(images),
        "Annotations": len(annotations),
        "Diagnosis categories": len(categories or {}),
        "Unknown image references": len(unknown_image_refs),
        "Missing referenced files": len(missing_files),
        "Invalid bounding boxes": len(invalid_boxes),
        "Duplicate image IDs": len(image_ids) - len(set(image_ids)),
        "Duplicate annotation IDs": len(annotation_ids) - len(set(annotation_ids)),
        "Unexpected class IDs": len(unexpected_class_ids),
    }

    errors = []
    if categories != expected_categories:
        errors.append(f"Unexpected disease categories: {categories}")
    for check in [
        "Unknown image references",
        "Missing referenced files",
        "Invalid bounding boxes",
        "Duplicate image IDs",
        "Duplicate annotation IDs",
        "Unexpected class IDs",
    ]:
        if checks[check]:
            errors.append(f"{check}: {checks[check]}")
    observed_classes = {int(annotation["category_id_3"]) for annotation in annotations}
    if observed_classes != set(expected_categories):
        errors.append("One or more expected disease classes have no annotations")
    if errors:
        raise RuntimeError("Selected dataset validation failed:\n- " + "\n- ".join(errors))

    report = pd.DataFrame({"Check": checks.keys(), "Value": checks.values()})
    class_table = pd.DataFrame(
        [{"Class ID": class_id, "Class name": name} for class_id, name in expected_categories.items()]
    )
    return report, class_table

In [7]:
source_validation, class_table = validate_source_dataset(
    source_payload,
    SOURCE_IMAGE_DIR,
    EXPECTED_CATEGORIES,
)
display(source_validation)
display(class_table)

,Check,Value
0,PNG files,705
1,JSON image records,705
2,Annotations,3529
3,Diagnosis categories,4
4,Unknown image references,0
5,Missing referenced files,0
6,Invalid bounding boxes,0
7,Duplicate image IDs,0
8,Duplicate annotation IDs,0
9,Unexpected class IDs,0


,Class ID,Class name
0,0,Impacted
1,1,Caries
2,2,Periapical Lesion
3,3,Deep Caries


## 6. Image-Level Multilabel Representation

The main ML task remains multiclass object detection: every annotated object has one diagnosis class and one bounding box. However, splitting is performed at image level, and one panoramic radiograph may contain several objects belonging to different classes. A temporary image-level multilabel representation is therefore constructed for split assignment.

For image $i$, the presence vector is

$$
\mathbf{y}_i =
(y_{i,0}, y_{i,1}, y_{i,2}, y_{i,3}),
\qquad
y_{i,c} \in \{0,1\},
$$

with

$$
y_{i,c} =
\begin{cases}
1, & \text{if class } c \text{ is present in image } i,\\
0, & \text{otherwise.}
\end{cases}
$$

For example, $\mathbf{y}_i=(0,1,1,0)$ means that image $i$ contains Caries and Periapical Lesion objects but contains neither Impacted nor Deep Caries objects. Multiple ones do not make the underlying task multilabel classification; they only record that separate objects of several classes coexist in one image.

The number of annotated objects, $N_i$, is retained in addition to class presence. Two images can have the same presence vector but very different annotation density. For example, both may contain Caries, while one contains a single Caries bounding box and the other contains several. Considering object count helps control not only whether classes appear in each split, but also how the total annotation volume is distributed.


In [8]:
def build_image_level_labels(
    images: list[dict],
    annotations: list[dict],
) -> pd.DataFrame:
    """Create one split-construction record per source image."""
    annotations_by_image = defaultdict(list)
    for annotation in annotations:
        annotations_by_image[int(annotation["image_id"])].append(annotation)

    rows = []
    for image in sorted(images, key=lambda item: int(item["id"])):
        image_id = int(image["id"])
        image_annotations = annotations_by_image[image_id]
        present = {int(annotation["category_id_3"]) for annotation in image_annotations}
        rows.append(
            {
                "image_id": image_id,
                "file_name": image["file_name"],
                "num_objects": len(image_annotations),
                "has_impacted": int(0 in present),
                "has_caries": int(1 in present),
                "has_periapical_lesion": int(2 in present),
                "has_deep_caries": int(3 in present),
            }
        )
    return pd.DataFrame(rows)

In [9]:
image_labels = build_image_level_labels(images, annotations)
multilabel_summary = (
    image_labels.drop(columns=["image_id", "file_name"])
    .sum()
    .rename("Total")
    .to_frame()
)
display(multilabel_summary)

,Total
num_objects,3529
has_impacted,254
has_caries,623
has_periapical_lesion,116
has_deep_caries,321


## 7. Train / Validation / Test Split

After validation and normalisation, the 705 images are divided in a 70/15/15 ratio:

$$
N_{\text{train}} \approx 0.70N, \qquad
N_{\text{val}} \approx 0.15N, \qquad
N_{\text{test}} \approx 0.15N,
\qquad N=705.
$$

The resulting exact sizes are 493 train images, 106 validation images, and 106 test images. Each subset has a distinct role. Train data is used to optimise model parameters. Validation data is used during experimentation to compare architectures and select hyperparameters, thresholds, augmentations, and other decisions that influence the final configuration. Test data is reserved for the final evaluation of the selected model.

The test subset is excluded from model fitting, architecture selection, hyperparameter tuning, and refinement. Aggregate dataset-level statistics may still be inspected for integrity and descriptive analysis. Using test metrics to choose an architecture or tune a configuration would allow information from that subset to influence model selection and would therefore compromise its role as an independent final estimate.

An ordinary random split is undesirable for this dataset. A radiograph may contain multiple annotated objects from several classes, the classes are imbalanced, and images differ in object count. A purely random assignment could produce substantially different class distributions across train, validation, and test. This is particularly risky for less represented categories, where moving only a small number of images can materially alter their validation or test coverage.

The notebook therefore implements a deterministic custom multilabel-aware greedy approximation. It is not presented as an exact implementation of a standard iterative-stratification algorithm. Images containing rarer labels are considered first, with class cardinality and object count contributing to ordering. For each image, candidate subsets are scored using their remaining class-label deficits, object-count deficit, and available capacity. The objective is to approach the exact subset sizes while preserving, as closely as practical, class presence, class co-occurrence patterns, and the volume of annotated objects.

Random tie-breaking uses the fixed seed `42`. The seed does not by itself improve split quality; its purpose is reproducibility. Given identical input data and parameters, repeated notebook execution produces the same image assignment rather than allowing experimental subsets to drift between runs.


In [10]:
def create_multilabel_split(
    image_labels: pd.DataFrame,
    split_sizes: dict[str, int],
    label_columns: list[str],
    random_seed: int,
) -> tuple[pd.DataFrame, dict[int, str]]:
    """Apply the existing deterministic greedy multilabel-aware assignment."""
    if sum(split_sizes.values()) != len(image_labels):
        raise RuntimeError(
            f"Split sizes total {sum(split_sizes.values())}, "
            f"but selected dataset has {len(image_labels)} images"
        )

    rng = random.Random(random_seed)
    working = image_labels.copy()
    working["_tie"] = [rng.random() for _ in range(len(working))]
    label_totals = working[label_columns].sum().to_dict()
    total_objects = int(working["num_objects"].sum())
    target_labels = {
        split: {
            label: label_totals[label] * size / len(working)
            for label in label_columns
        }
        for split, size in split_sizes.items()
    }
    target_objects = {
        split: total_objects * size / len(working)
        for split, size in split_sizes.items()
    }

    def rarity_score(row):
        active = [label_totals[label] for label in label_columns if row[label]]
        return min(active) if active else float("inf")

    working["_rarity"] = working.apply(rarity_score, axis=1)
    working["_cardinality"] = working[label_columns].sum(axis=1)
    working = working.sort_values(
        ["_rarity", "_cardinality", "num_objects", "_tie"],
        ascending=[True, False, False, True],
    )

    assignments = {}
    split_counts = Counter()
    split_label_counts = {split: Counter() for split in split_sizes}
    split_object_counts = Counter()
    split_ties = {split: rng.random() for split in split_sizes}

    for _, row in working.iterrows():
        candidates = [
            split
            for split, size in split_sizes.items()
            if split_counts[split] < size
        ]
        active_labels = [label for label in label_columns if int(row[label]) == 1]

        def assignment_score(split):
            label_deficits = [
                (target_labels[split][label] - split_label_counts[split][label])
                / max(target_labels[split][label], 1)
                for label in active_labels
            ]
            label_score = sum(label_deficits) / len(label_deficits) if label_deficits else 0.0
            object_score = (
                target_objects[split] - split_object_counts[split]
            ) / max(target_objects[split], 1)
            capacity_score = (
                split_sizes[split] - split_counts[split]
            ) / split_sizes[split]
            return (
                2.0 * label_score + 0.35 * object_score + 0.5 * capacity_score,
                split_ties[split],
            )

        chosen = max(candidates, key=assignment_score)
        image_id = int(row["image_id"])
        assignments[image_id] = chosen
        split_counts[chosen] += 1
        split_object_counts[chosen] += int(row["num_objects"])
        for label in active_labels:
            split_label_counts[chosen][label] += 1

    result = image_labels.copy()
    result["split"] = result["image_id"].map(assignments)
    if result["split"].isna().any() or result["image_id"].duplicated().any():
        raise RuntimeError("Split assignment is incomplete or duplicated")
    if result["split"].value_counts().to_dict() != split_sizes:
        raise RuntimeError(f"Unexpected split sizes: {result['split'].value_counts().to_dict()}")
    return result, assignments

In [11]:
image_labels, split_assignments = create_multilabel_split(
    image_labels,
    SPLIT_SIZES,
    LABEL_COLUMNS,
    RANDOM_SEED,
)
assignments = split_assignments
split_size_table = (
    image_labels["split"]
    .value_counts()
    .reindex(SPLIT_SIZES)
    .rename_axis("Split")
    .to_frame("Images")
)
display(split_size_table)

,Images
Split,
train,493
val,106
test,106


## 8. Patient-Level Leakage Limitation

For medical imaging, the strongest split would normally be patient-level: all images belonging to one patient would be assigned to exactly one subset. This prevents related observations from the same patient from appearing in both development and evaluation data, for example in train and test.

The available DENTEX image and annotation records expose no reliable `patient_id`, `case_id`, `study_id`, or `subject_id`. The filenames are generic image identifiers and do not provide a documented patient grouping. Consequently, this project must use an image-level split and cannot guarantee patient-level independence from the available metadata. This is a dataset limitation that must be considered when interpreting model results.


## 9. Split Distribution Review

Once assignment is complete, the notebook verifies the exact image counts and reviews two complementary class distributions. The object-instance table counts every annotated bounding box of each class. The image-presence table counts the number of images in which each class occurs, regardless of how many instances of that class the image contains.

Both views are required. Instance counts describe the amount of object-level supervision available to the detector, while presence counts reflect the image-level quantities used by the multilabel-aware split. The review also confirms that all four diagnosis categories occur in each subset. This is especially important for validation and test: if a class were absent, its detection performance could not be evaluated meaningfully on that subset.

The warning check compares observed image-level presence with the proportional expectation for each split and flags severe deviations or a missing class. It is a diagnostic review of the deterministic assignment, not a second splitting stage. Detailed study of class imbalance, object density, and geometric properties belongs to the subsequent EDA notebook.


In [12]:
def summarize_split_distribution(
    image_labels: pd.DataFrame,
    annotations: list[dict],
    assignments: dict[int, str],
    expected_categories: dict[int, str],
    split_sizes: dict[str, int],
    label_columns: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Summarise object instances and image-level class presence by split."""
    instance_rows = {"Images": Counter(image_labels["split"])}
    for class_id, class_name in expected_categories.items():
        counts = Counter()
        for annotation in annotations:
            if int(annotation["category_id_3"]) == class_id:
                counts[assignments[int(annotation["image_id"])]] += 1
        instance_rows[f"{class_name} instances"] = counts

    instance_table = pd.DataFrame(
        {
            split.title(): {metric: counts[split] for metric, counts in instance_rows.items()}
            for split in split_sizes
        }
    )
    instance_table["Total"] = instance_table.sum(axis=1)

    presence_names = {
        "has_impacted": "Images with Impacted",
        "has_caries": "Images with Caries",
        "has_periapical_lesion": "Images with Periapical Lesion",
        "has_deep_caries": "Images with Deep Caries",
    }
    presence_table = pd.DataFrame(
        {
            split.title(): {
                presence_names[label]: int(
                    image_labels.loc[image_labels["split"] == split, label].sum()
                )
                for label in label_columns
            }
            for split in split_sizes
        }
    )

    review_warnings = []
    for label, display_name in presence_names.items():
        total = int(image_labels[label].sum())
        for split, target_size in split_sizes.items():
            actual = int(image_labels.loc[image_labels["split"] == split, label].sum())
            expected = total * target_size / len(image_labels)
            relative_error = abs(actual - expected) / max(expected, 1)
            if actual == 0 or relative_error > 0.35:
                review_warnings.append(
                    f"{display_name} may be underrepresented in {split}: "
                    f"actual={actual}, proportional expectation={expected:.1f}"
                )
    return instance_table, presence_table, review_warnings

In [13]:
instance_table, presence_table, distribution_warnings = summarize_split_distribution(
    image_labels,
    annotations,
    split_assignments,
    EXPECTED_CATEGORIES,
    SPLIT_SIZES,
    LABEL_COLUMNS,
)
display(instance_table)
display(presence_table)
for message in distribution_warnings:
    warnings.warn(message)

,Train,Val,Test,Total
Images,493,106,106,705
Impacted instances,409,99,96,604
Caries instances,1501,355,333,2189
Periapical Lesion instances,109,23,26,158
Deep Caries instances,423,71,84,578


,Train,Val,Test
Images with Impacted,178,38,38
Images with Caries,436,93,94
Images with Periapical Lesion,80,18,18
Images with Deep Caries,225,48,48


## 10. Processed COCO Dataset Creation

After selection, validation, and split assignment, the hierarchical DENTEX annotation is transformed into the simpler schema required by this four-class object-detection project. In the original fully annotated data, `category_id_1`, `category_id_2`, and `category_id_3` represent different levels of the DENTEX hierarchy. The first two levels are not target variables for the present task; the diagnosis label is the third level.

The derived annotation therefore applies the explicit mapping

$$
\texttt{category\_id\_3} \longrightarrow \texttt{category\_id},
$$

and removes `category_id_1` and `category_id_2`. Each processed object then has one unambiguous class label directly aligned with the project task. The processed `categories` section contains only the four diagnosis classes.

Original image IDs and annotation IDs are not renumbered. They are already unique, and preserving them maintains traceability between each processed record and its source DENTEX record. The source data under `raw/` is not modified; selected PNG files are copied into the derived dataset.

The canonical output structure is:

```text
data/processed/dentex_diagnosis/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
├── annotations/
│   ├── train.json
│   ├── val.json
│   └── test.json
└── split_summary.csv
```

Each split JSON contains only its own image records and annotations, together with the four shared category definitions. `split_summary.csv` records the assigned split, object count, and image-level class-presence indicators for every source image. It freezes the deterministic assignment so that later experiments can use the same partition consistently.


In [14]:
def build_processed_payload(
    split: str,
    image_labels: pd.DataFrame,
    images: list[dict],
    annotations: list[dict],
    expected_categories: dict[int, str],
) -> tuple[dict, set[int]]:
    """Convert one split from hierarchical DENTEX labels to project COCO labels."""
    split_ids = set(
        image_labels.loc[image_labels["split"] == split, "image_id"].astype(int)
    )
    split_images = [image for image in images if int(image["id"]) in split_ids]
    split_annotations = []
    for source_annotation in annotations:
        if int(source_annotation["image_id"]) not in split_ids:
            continue
        annotation = dict(source_annotation)
        annotation["category_id"] = int(annotation.pop("category_id_3"))
        annotation.pop("category_id_1", None)
        annotation.pop("category_id_2", None)
        split_annotations.append(annotation)

    categories = [
        {"id": class_id, "name": class_name}
        for class_id, class_name in expected_categories.items()
    ]
    return {
        "images": split_images,
        "annotations": split_annotations,
        "categories": categories,
    }, split_ids


def write_processed_dataset(
    processed_root: Path,
    source_image_dir: Path,
    image_labels: pd.DataFrame,
    images: list[dict],
    annotations: list[dict],
    expected_categories: dict[int, str],
    split_sizes: dict[str, int],
) -> tuple[Path, Path, pd.DataFrame]:
    """Write the canonical processed images, COCO JSON files, and split summary."""
    image_output_root = processed_root / "images"
    annotation_output_root = processed_root / "annotations"
    processed_root.mkdir(parents=True, exist_ok=True)
    annotation_output_root.mkdir(parents=True, exist_ok=True)

    for split in split_sizes:
        output_image_dir = image_output_root / split
        if output_image_dir.exists():
            shutil.rmtree(output_image_dir)
        output_image_dir.mkdir(parents=True)

        payload, _ = build_processed_payload(
            split,
            image_labels,
            images,
            annotations,
            expected_categories,
        )
        for image in payload["images"]:
            shutil.copy2(
                source_image_dir / image["file_name"],
                output_image_dir / image["file_name"],
            )
        with (annotation_output_root / f"{split}.json").open("w", encoding="utf-8") as handle:
            json.dump(payload, handle, indent=2, ensure_ascii=False)

    summary_columns = [
        "image_id",
        "file_name",
        "split",
        "num_objects",
        "has_impacted",
        "has_caries",
        "has_periapical_lesion",
        "has_deep_caries",
    ]
    split_summary = image_labels[summary_columns].sort_values(["split", "image_id"])
    split_summary.to_csv(processed_root / "split_summary.csv", index=False)
    return image_output_root, annotation_output_root, split_summary

In [15]:
IMAGE_OUTPUT_ROOT, ANNOTATION_OUTPUT_ROOT, split_summary = write_processed_dataset(
    PROCESSED_ROOT,
    SOURCE_IMAGE_DIR,
    image_labels,
    images,
    annotations,
    EXPECTED_CATEGORIES,
    SPLIT_SIZES,
)

output_paths = pd.DataFrame(
    {
        "Artifact": ["Split summary", "Train annotations", "Validation annotations", "Test annotations"],
        "Path": [
            PROCESSED_ROOT / "split_summary.csv",
            ANNOTATION_OUTPUT_ROOT / "train.json",
            ANNOTATION_OUTPUT_ROOT / "val.json",
            ANNOTATION_OUTPUT_ROOT / "test.json",
        ],
    }
)
display(output_paths)

,Artifact,Path
0,Split summary,C:\Users\Eclipse\PycharmProjects\Dental-Xray-P...
1,Train annotations,C:\Users\Eclipse\PycharmProjects\Dental-Xray-P...
2,Validation annotations,C:\Users\Eclipse\PycharmProjects\Dental-Xray-P...
3,Test annotations,C:\Users\Eclipse\PycharmProjects\Dental-Xray-P...


## 11. Final Dataset Verification

The final verification checks the generated dataset rather than assuming that successful file writing implies correctness. First, all 705 source images must be assigned exactly once. The union of the three subsets must reproduce the complete selected source set:

$$
D_{\text{train}} \cup D_{\text{val}} \cup D_{\text{test}} = D,
\qquad |D|=705.
$$

At the same time, the subsets must be pairwise disjoint:

$$
D_{\text{train}} \cap D_{\text{val}} = \varnothing,
$$

$$
D_{\text{train}} \cap D_{\text{test}} = \varnothing,
$$

$$
D_{\text{val}} \cap D_{\text{test}} = \varnothing.
$$

This ensures that each source image occurs in one and only one split. Annotation preservation is checked similarly: the combined processed annotation IDs must match the source annotation IDs exactly, with no loss, duplication, or cross-split reuse.

Within each split, every `annotation.image_id` must refer to an image record in that same split. Formally, for

$$
s \in \{\text{train},\text{val},\text{test}\},
$$

the requirement is

$$
\forall a \in A_s:\quad a.\texttt{image\_id} \in I_s,
$$

where $A_s$ and $I_s$ denote the annotation and image-ID sets for subset $s$. This prevents cross-split or missing image references.

JSON-to-filesystem consistency is checked in both directions. Every image record must have a physical PNG in the corresponding `images/<split>/` directory, and the set of PNG filenames on disk must exactly equal the `file_name` values in that split's JSON. Each split must also contain annotations for all four target classes. Complete class coverage is particularly important in validation and test because an absent class would make class-specific evaluation impossible.

Finally, the notebook confirms that `split_summary.csv` contains one unique row for every selected source image. This file persists the split membership used by subsequent EDA, training, comparison, and evaluation stages. The final compact table reports image, annotation, and class counts for every split and their totals.


In [16]:
def validate_processed_dataset(
    image_output_root: Path,
    annotation_output_root: Path,
    split_summary: pd.DataFrame,
    source_image_ids: list[int],
    source_annotation_ids: list[int],
    expected_categories: dict[int, str],
    split_sizes: dict[str, int],
) -> pd.DataFrame:
    """Validate generated files, membership, references, annotations, and classes."""
    processed_ids = {}
    processed_annotation_ids = []
    rows = []

    for split, expected_size in split_sizes.items():
        json_path = annotation_output_root / f"{split}.json"
        image_dir = image_output_root / split
        if not json_path.is_file() or not image_dir.is_dir():
            raise RuntimeError(f"Missing processed output for split: {split}")

        with json_path.open("r", encoding="utf-8") as handle:
            payload = json.load(handle)
        ids = {int(image["id"]) for image in payload["images"]}
        annotation_image_ids = {
            int(annotation["image_id"]) for annotation in payload["annotations"]
        }
        file_names = {image["file_name"] for image in payload["images"]}
        disk_names = {path.name for path in image_dir.glob("*.png")}
        class_ids = {
            int(annotation["category_id"]) for annotation in payload["annotations"]
        }

        if len(ids) != expected_size or len(file_names) != expected_size:
            raise RuntimeError(f"Unexpected image count in {split}")
        if disk_names != file_names:
            raise RuntimeError(f"Physical PNG/JSON mismatch in {split}")
        if not annotation_image_ids.issubset(ids):
            raise RuntimeError(f"Cross-split or missing image reference in {split}")
        if class_ids != set(expected_categories):
            raise RuntimeError(f"Not all diagnosis classes are represented in {split}: {class_ids}")

        processed_ids[split] = ids
        processed_annotation_ids.extend(
            int(annotation["id"]) for annotation in payload["annotations"]
        )
        rows.append(
            {
                "split": split,
                "images": len(ids),
                "annotations": len(payload["annotations"]),
                "classes": len(class_ids),
            }
        )

    split_names = list(split_sizes)
    for index, left in enumerate(split_names):
        for right in split_names[index + 1:]:
            if processed_ids[left] & processed_ids[right]:
                raise RuntimeError(f"Images overlap between {left} and {right}")

    all_processed_ids = set().union(*processed_ids.values())
    if all_processed_ids != set(source_image_ids):
        raise RuntimeError("The selected 705 images were not assigned exactly once")
    if Counter(processed_annotation_ids) != Counter(source_annotation_ids):
        raise RuntimeError("Source annotations were not preserved exactly once")
    if len(split_summary) != len(source_image_ids):
        raise RuntimeError("split_summary.csv does not contain every source image")
    if split_summary["image_id"].nunique() != len(source_image_ids):
        raise RuntimeError("split_summary.csv contains duplicate image IDs")

    report = pd.DataFrame(rows).set_index("split")
    report.loc["total"] = report.sum(numeric_only=True)
    return report

In [17]:
image_ids = [int(image["id"]) for image in images]
annotation_ids = [int(annotation["id"]) for annotation in annotations]

final_verification = validate_processed_dataset(
    IMAGE_OUTPUT_ROOT,
    ANNOTATION_OUTPUT_ROOT,
    split_summary,
    image_ids,
    annotation_ids,
    EXPECTED_CATEGORIES,
    SPLIT_SIZES,
)
display(final_verification)

,images,annotations,classes
split,,,
train,493,2442,4
val,106,548,4
test,106,539,4
total,705,3529,12


## 12. Dataset Preparation Summary

The fully annotated 705-image DENTEX diagnosis subset has been selected by annotation content, validated before splitting, normalised from the hierarchical source labels to the project-specific COCO-compatible schema, divided reproducibly into 493 train, 106 validation, and 106 test images, and verified against both source records and generated files. The immutable `raw/` dataset remains unchanged. The canonical processed dataset is ready for exploratory data analysis.
